In [1]:
import pandas as pd

column_names = ['Movie_ID', 'User_ID', 'Rating', 'Date']

data = []
file_path = 'combined_data_1.txt'

with open(file_path, 'r', encoding='latin1') as f:
    current_movie_id = None
    for line in f:
        line = line.strip()
        if line.endswith(':'):
            current_movie_id = line[:-1]
        else:
            parts = line.split(',')
            if len(parts) == 3:
                data.append([current_movie_id, parts[0], parts[1], parts[2]])

# Ստեղծում ենք աղյուսակը և տալիս անուններ
df = pd.DataFrame(data, columns=column_names)

df['Movie_ID'] = df['Movie_ID'].astype(int)
df['User_ID'] = df['User_ID'].astype(int)
df['Rating'] = df['Rating'].astype(float)
df

,Movie_ID,User_ID,Rating,Date
0,1,1488844,3.0,2005-09-06
1,1,822109,5.0,2005-05-13
2,1,885013,4.0,2005-10-19
3,1,30878,4.0,2005-12-26
4,1,823519,3.0,2004-05-03
...,...,...,...,...
24053759,4499,2591364,2.0,2005-02-16
24053760,4499,1791000,2.0,2005-02-10
24053761,4499,512536,5.0,2005-07-27
24053762,4499,988963,3.0,2005-12-20


In [2]:
import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix

import implicit

import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

/home/honor/Programing/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
def create_x(df):
    M = df['Movie_ID'].nunique()
    N = df['User_ID'].nunique()

    user_mapper = dict(zip(np.unique(df["User_ID"]), list(range(N))))
    movie_mapper = dict(zip(np.unique(df["Movie_ID"]), list(range(M))))

    user_inv_mapper = dict(zip(list(range(N)), np.unique(df["User_ID"])))
    movie_inv_mapper = dict(zip(list(range(M)), np.unique(df["Movie_ID"])))

    user_index = [user_mapper[i] for i in df['User_ID']]
    movie_index = [movie_mapper[i] for i in df['Movie_ID']]


    X = csr_matrix((df["Rating"], (movie_index, user_index)), shape=(M, N))
        
    return X, user_mapper, movie_mapper, user_inv_mapper, movie_inv_mapper

In [4]:
X, user_mapper, movie_mapper, user_inv_mapper, movie_inv_mapper = create_x(df)

In [5]:
import implicit.evaluation
from implicit.nearest_neighbours import CosineRecommender, BM25Recommender
import pandas as pd


In [6]:
user_item_matrix = X.T.tocsr()
train_data, test_data = implicit.evaluation.train_test_split(
    user_item_matrix, 
    train_percentage=0.8, 
    random_state=42
)

als_model = implicit.als.AlternatingLeastSquares(factors=20, regularization=0.01, random_state=42)
als_model.fit(train_data)

knn_model = BM25Recommender(K=300)
knn_model.fit(train_data)

als_p_at_k = implicit.evaluation.precision_at_k(als_model, train_data, test_data, K=10)
als_map_at_k = implicit.evaluation.mean_average_precision_at_k(als_model, train_data, test_data, K=10)

knn_p_at_k = implicit.evaluation.precision_at_k(knn_model, train_data, test_data, K=10)
knn_map_at_k = implicit.evaluation.mean_average_precision_at_k(knn_model, train_data, test_data, K=10)

results_df = pd.DataFrame({
    'Model / Մոդել': ['Alternating Least Squares (ALS)', 'K-Nearest Neighbors (Cosine)'],
    'Precision@10': [als_p_at_k, knn_p_at_k],
    'MAP@10': [als_map_at_k, knn_map_at_k]
})

display(results_df)




/home/honor/Programing/.venv/lib/python3.12/site-packages/implicit/cpu/als.py:96: RuntimeWarning: OpenBLAS is configured to use 18 threads. It is highly recommended to disable its internal threadpool by setting the environment variable 'OPENBLAS_NUM_THREADS=1' or by calling 'threadpoolctl.threadpool_limits(1, "blas")'. Having OpenBLAS use a threadpool can lead to severe performance issues here.
  check_blas_config()
100%|██████████| 15/15 [01:02<00:00,  4.18s/it]
/home/honor/Programing/.venv/lib/python3.12/site-packages/implicit/utils.py:164: ParameterWarning: Method expects CSR input, and was passed coo_matrix instead. Converting to CSR took 0.15233588218688965 seconds
  warnings.warn(
100%|██████████| 413076/413076 [00:27<00:00, 14899.53it/s]


,Model / Մոդել,Precision@10,MAP@10
0,Alternating Least Squares (ALS),0.350477,0.184311
1,K-Nearest Neighbors (Cosine),0.296980,0.150232


In [ ]:
all_user_ids = np.arange(train_data.shape[0])

als_ids, _ = als_model.recommend(all_user_ids, train_data, N=10, filter_already_liked_items=True)
knn_ids, _ = knn_model.recommend(all_user_ids, train_data, N=10, filter_already_liked_items=True)

total_items = train_data.shape[1]

als_unique_items_recommended = len(np.unique(als_ids))
knn_unique_items_recommended = len(np.unique(knn_ids))

als_coverage = als_unique_items_recommended / total_items
knn_coverage = knn_unique_items_recommended / total_items

total_users = train_data.shape[0]

item_interactions = np.bincount(train_data.indices, minlength=total_items)

item_probabilities = np.maximum(item_interactions / total_users, 1e-9)

def calculate_novelty(recommendation_matrix, item_probs):
    
    recommended_probs = item_probs[recommendation_matrix]
    
    novelty_scores = -np.log2(recommended_probs)
    
    return np.mean(novelty_scores)

als_novelty = calculate_novelty(als_ids, item_probabilities)
knn_novelty = calculate_novelty(knn_ids, item_probabilities)

advanced_results_df = pd.DataFrame({
    'Model / Մոդել': ['Alternating Least Squares (ALS)', 'K-Nearest Neighbors (Cosine)'],
    'Precision@10': [als_p_at_k, knn_p_at_k],
    'MAP@10': [als_map_at_k, knn_map_at_k],
    'Catalog Coverage / Ծածկույթ': [als_coverage, knn_coverage],
    'Novelty / Նորույթ': [als_novelty, knn_novelty]
})

print("\n--- Final Model Evaluation Metrics ---")
display(advanced_results_df)


--- Final Model Evaluation Metrics ---


,Model / Մոդել,Precision@10,MAP@10,Catalog Coverage / Ծածկույթ,Novelty / Նորույթ
0,Alternating Least Squares (ALS),0.350477,0.184311,0.140476,2.769020
1,K-Nearest Neighbors (Cosine),0.296980,0.150232,0.401645,2.820391
